Apache Spark
-----

Apache Spark is an open-source, distributed computing system that provides a fast and general-purpose cluster-computing framework for big data processing. It was developed to address the limitations and challenges posed by traditional MapReduce-based systems by introducing a more flexible and efficient architecture. The goal of this assignment is to familiarize you with Apache Spark operations through practical exercises.

In [1]:
# importing required libraries
import os
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, substring, avg, mean, sum, round, count, collect_list, last, when, lower, regexp_replace, current_date, datediff, when, to_date, expr, format_number,trim, desc, first
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, BooleanType, FloatType, DoubleType, TimestampType, IntegerType, LongType
from pyspark.ml.feature import Imputer, VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

In [2]:
# Ensure the dataset is present (creates ./dataset/*.csv)
from downloader import ensure_dataset

ensure_dataset()


Extracting CSVs into ./dataset/ ...
Done. CSVs are in: /uufs/chpc.utah.edu/common/home/u1564808/Documents/CS-6964-Spring-2026-Lab-2/dataset


PosixPath('/uufs/chpc.utah.edu/common/home/u1564808/Documents/CS-6964-Spring-2026-Lab-2/dataset')

In [3]:
!echo $SPARK_MASTER_ADDRESS

spark://notch315.ipoib.int.chpc.utah.edu:7081


In [4]:
spark = SparkSession.builder.master(os.getenv('SPARK_MASTER_ADDRESS')).appName('Spark-application').config('spark.ui.enabled', 'false').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/21 17:23:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!echo $SPARK_MASTER_OOD_ADDRESS
!echo $SPARK_WORKER1_OOD_ADDRESS
!echo $SPARK_WORKER2_OOD_ADDRESS

https://ondemand-class.chpc.utah.edu/rnode/notch315.ipoib.int.chpc.utah.edu/8087
https://ondemand-class.chpc.utah.edu/rnode/notch315.ipoib.int.chpc.utah.edu/8088
https://ondemand-class.chpc.utah.edu/rnode/notch316.ipoib.int.chpc.utah.edu/8086


# Note: 

1. We have provided many hints on which functions to use for a particular task. If you feel like you can solve it without those methods or using different methods, you can do so.

2. Also, if you feel like that some columns should have been type converted or dropped and are not done during this exercise, feel free to do so. You can provide a small explanation.

3. The feature columns should be the following columns. Some are already present in the dataset and some are derived below in the exercise:

    'investment_rounds',
    'invested_companies',
    'funding_rounds',
    'relationships',
    'age_of_company',
    'total_amount_raised',
    'num_acquisitions',
    'have_been_acquired',
    'fin_org_financed',
    'person_financed',
    'startup_financed',
    'num_products',
    'category_code_index',
    'country_code_index'

4. If before creating feature vector, you have other columns in your dataset than mentioned above, drop them. If you want, you can include them in features, but mention it in the notebook.

5. Your code should be error free as we will run each cell while grading.

## Dataset

The database is composed of 5 tables containing many aspects related to the startup world from 1901-01-01 to 2014-10-01. Each table in the dataset represents a different aspect of the startup ecosystem, detailing the interactions, events, and entities involved in startup funding, growth, and development.

-------
### Objects

A broad table that likely serves as a central repository for entities in the dataset, including companies, startups, and perhaps other organizational forms. It includes detailed information on each entity, such as its status, industry category, funding received, and key milestones.
It contains 40 variables, of which the most important are name, entity_type, category_code, status, founded_at, country_code, state_code, investment_rounds, invested_companies, funding_rounds.

------
### Investments

Tracks investment transactions, detailing which entities (investors) have invested in which companies or startups during particular funding rounds.


-------
<!-- Offices
-----
geographic position of main offices (both of the companies and the investment funds). -->

### Funding Rounds

Captures details about specific funding events where startups or companies receive investment. (funded company, date and funding type, total raised amount, number of participants)

-----

### Relationships

Tracks the connections between individuals (people) and entities (companies, startups), including roles or positions held by individuals within companies, indicating the network of professionals in the startup ecosystem. (people, institutions, start and end date of relationship, role held)

------
### Acquisitions

Details about acquisition events where one company purchases another. (acquired company, acquiring company, price and date of acquisition, payment method)

-----

Here are some links to documentation which would be helpful for the tasks:


Pyspark sql datframe: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html


Pyspark sql functions: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html



# Exercise

## 1 .Read the following files to the Spark DataFrame and select given columns


1. **objects.csv**

   -  **Columns:** "id",
    "entity_type",
    "entity_id",
    "parent_id",
    "name",
    "category_code",
    "status",
    "founded_at",
    "closed_at",
    "country_code",
    "state_code",
    "city",
    "region",
    "investment_rounds",
    "invested_companies",
    "first_funding_at",
    "last_funding_at",
    "funding_rounds",
    "funding_total_usd",
    "participants",
    "relationships"
  
2. **acquisitions.csv**

   - **Columns:** 'id', 'acquisition_id', 'acquiring_object_id', 'acquired_object_id',
       'term_code', 'price_amount', 'price_currency_code', 'acquired_at'
  
2. **funding_rounds.csv**

   - **Columns:** "object_id",
    "funded_at",
    "funding_round_type",
    "funding_round_code",
    "raised_amount_usd",
    "is_first_round",
    "is_last_round"
    
2. **investments.csv**

   - **Columns:** 'funding_round_id', 'funded_object_id', 'investor_object_id'
  
2. **relationships.csv**

   - **Columns:** "id",
    "person_object_id",
    "relationship_object_id",
    "start_at",
    "end_at",
    "is_past",
    "title"

##### Read following csv files to the spark dataframe

Hint: Use .select() to select the required columns

In [6]:
#objects
# milestones column added as well for 2.1.1
objects = spark.read.csv("./dataset/objects.csv", header=True, inferSchema=True).select(
    "id", "entity_type", "entity_id", "parent_id", "name", "category_code",
    "status", "founded_at", "closed_at", "country_code", "state_code", "city",
    "region", "investment_rounds", "invested_companies", "first_funding_at",
    "last_funding_at", "funding_rounds", "funding_total_usd", "participants","milestones", "relationships"
)


In [7]:
#acquisitions
acquisitions = spark.read.csv("./dataset/acquisitions.csv", header=True, inferSchema=True).select(
    "id", "acquisition_id", "acquiring_object_id", "acquired_object_id",
    "term_code", "price_amount", "price_currency_code", "acquired_at"
)


In [8]:
#funding_rounds
funding_rounds = spark.read.csv("./dataset/funding_rounds.csv", header=True, inferSchema=True).select(
    "object_id", "funded_at", "funding_round_type", "funding_round_code",
    "raised_amount_usd", "is_first_round", "is_last_round"
)


In [9]:
# investments
investments = spark.read.csv("./dataset/investments.csv", header=True, inferSchema=True).select(
    "funding_round_id", "funded_object_id", "investor_object_id"
)


In [10]:
#relationships
relationships = spark.read.csv("./dataset/relationships.csv", header=True, inferSchema=True).select(
    "id", "person_object_id", "relationship_object_id",
    "start_at", "end_at", "is_past", "title"
)


## 2.1 Clean objects table


1. Type conversion: convert the datatype of given columns
        
        Column with dates: datetime type

        investment_rounds, invested_companies, funding_rounds, funding_total_usd, milestones, relationships: Numeric


2. Find Age of company in years.

        
        To calculate the age of a company, subtract the date in the 'founded_at' column from the date in the 'closed_at' column. If the 'closed_at' date is not provided (NULL), use the current date(today's date) instead.  If 'founded_at' date is NULL, then age of company will be NULL.
        After the above operation fill the NULL values in 'age_of_company' column with median age.
        Drop 'founded_at' and 'closed_at' after deriving the new column.


3. Handle Variation. 

        The 'status' column in objects dataframe categorizes a company's current operational phase, such as 'operating', 'ipo', 'acquired', or 'closed', reflecting its lifecycle stage.
        
        To ensure uniformity in company status reporting, categorize all entries in the 'status' column into one of four standardized categories: 'operating', 'ipo', 'acquired', or 'closed'. You need to map various existing status names, which might currently reflect the same operational state in different terminology. Display the count of each status after modification. 





Hint: Use methods withColumn, col, cast for type conversion.

In [11]:
#1.1 Type conversion to timestamp
objects.printSchema()

# check the data for datetime type.
objects.show(20)

#date columns are "founded_at","first_funding_at","last_funding_at","closed_at"
# closed_at is already datetime type, we don't include it.
date_columns = ["founded_at","first_funding_at","last_funding_at"]

for col_name in date_columns:
    objects = objects.withColumn(col_name, to_date(col(col_name)))


root
 |-- id: string (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- entity_id: integer (nullable = true)
 |-- parent_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- status: string (nullable = true)
 |-- founded_at: string (nullable = true)
 |-- closed_at: date (nullable = true)
 |-- country_code: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- investment_rounds: integer (nullable = true)
 |-- invested_companies: integer (nullable = true)
 |-- first_funding_at: string (nullable = true)
 |-- last_funding_at: string (nullable = true)
 |-- funding_rounds: string (nullable = true)
 |-- funding_total_usd: string (nullable = true)
 |-- participants: string (nullable = true)
 |-- milestones: string (nullable = true)
 |-- relationships: string (nullable = true)

+--------+-----------+---------+---------+----------

In [12]:
objects.show(20)

+--------+-----------+---------+---------+--------------------+---------------+-----------+----------+----------+------------+----------+-------------+-----------+-----------------+------------------+----------------+---------------+--------------+-----------------+------------+----------+-------------+
|      id|entity_type|entity_id|parent_id|                name|  category_code|     status|founded_at| closed_at|country_code|state_code|         city|     region|investment_rounds|invested_companies|first_funding_at|last_funding_at|funding_rounds|funding_total_usd|participants|milestones|relationships|
+--------+-----------+---------+---------+--------------------+---------------+-----------+----------+----------+------------+----------+-------------+-----------+-----------------+------------------+----------------+---------------+--------------+-----------------+------------+----------+-------------+
|     c:1|    Company|        1|     NULL|            Wetpaint|            web|Operat

In [13]:
#1.2 Type conversion to integer
#investment_rounds, invested_companies are already integer type. But for the rest, converted to double.
numeric_columns = ["funding_rounds", "funding_total_usd", "milestones", "relationships"]

for col_name in numeric_columns:
    objects = objects.withColumn(col_name, col(col_name).cast(DoubleType()))

objects.printSchema()


root
 |-- id: string (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- entity_id: integer (nullable = true)
 |-- parent_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- status: string (nullable = true)
 |-- founded_at: date (nullable = true)
 |-- closed_at: date (nullable = true)
 |-- country_code: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- investment_rounds: integer (nullable = true)
 |-- invested_companies: integer (nullable = true)
 |-- first_funding_at: date (nullable = true)
 |-- last_funding_at: date (nullable = true)
 |-- funding_rounds: double (nullable = true)
 |-- funding_total_usd: double (nullable = true)
 |-- participants: string (nullable = true)
 |-- milestones: double (nullable = true)
 |-- relationships: double (nullable = true)



Hint: Some function used will be when, col, datediff, currentdate

In [14]:
# 2. Age_of_company
"""
 To calculate the age of a company, subtract the date in the 'founded_at' column from the date in the 'closed_at' column. If the 'closed_at' date is not provided (NULL), use the current date(today's date) instead.  If 'founded_at' date is NULL, then age of company will be NULL.
 After the above operation fill the NULL values in 'age_of_company' column with median age.
 Drop 'founded_at' and 'closed_at' after deriving the new column.
"""
objects.select("founded_at", "closed_at").show(10)

+----------+---------+
|founded_at|closed_at|
+----------+---------+
|2005-10-17|     NULL|
|      NULL|     NULL|
|      NULL|     NULL|
|2008-07-26|     NULL|
|2008-07-26|     NULL|
|2007-06-27|     NULL|
|2008-08-20|     NULL|
|      NULL|     NULL|
|      NULL|     NULL|
|2011-08-01|     NULL|
+----------+---------+
only showing top 10 rows



In [15]:
#datediff returns days, divided by 365
objects = objects.withColumn("age_of_company",when(col("founded_at").isNull(), None).otherwise(datediff(when(col("closed_at").isNull(), current_date()).otherwise(col("closed_at")),col("founded_at")) / 365))

objects.select("founded_at", "closed_at", "age_of_company").show(10)

# Fill nulls with median age
median_age = objects.approxQuantile("age_of_company", [0.5], 0.001)[0]
objects = objects.withColumn("age_of_company",when(col("age_of_company").isNull(), median_age).otherwise(col("age_of_company"))
)

+----------+---------+------------------+
|founded_at|closed_at|    age_of_company|
+----------+---------+------------------+
|2005-10-17|     NULL|20.361643835616437|
|      NULL|     NULL|              NULL|
|      NULL|     NULL|              NULL|
|2008-07-26|     NULL|17.586301369863012|
|2008-07-26|     NULL|17.586301369863012|
|2007-06-27|     NULL| 18.66849315068493|
|2008-08-20|     NULL|17.517808219178082|
|      NULL|     NULL|              NULL|
|      NULL|     NULL|              NULL|
|2011-08-01|     NULL| 14.56986301369863|
+----------+---------+------------------+
only showing top 10 rows



In [16]:
# Drop founded_at and closed_at
objects = objects.drop("founded_at", "closed_at")

In [17]:
objects.select("age_of_company").show(10)

+------------------+
|    age_of_company|
+------------------+
|20.361643835616437|
| 17.15068493150685|
| 17.15068493150685|
|17.586301369863012|
|17.586301369863012|
| 18.66849315068493|
|17.517808219178082|
| 17.15068493150685|
| 17.15068493150685|
| 14.56986301369863|
+------------------+
only showing top 10 rows



In [18]:
# 3. Map status to predefined categories
"""
 The 'status' column in objects dataframe categorizes a company's current operational phase,
 such as 'operating', 'ipo', 'acquired', or 'closed', reflecting its lifecycle stage.
 To ensure uniformity in company status reporting, categorize all entries in the 'status' column into one of four standardized categories:
 'operating', 'ipo', 'acquired', or 'closed'. You need to map various existing status names,
 which might currently reflect the same operational state in different terminology.
 Display the count of each status after modification.
"""

# First check 'status' column
objects.groupBy("status").count().orderBy("count", ascending=False).show(50)


+-----------+------+
|     status| count|
+-----------+------+
|    operate|138753|
|   operates|138685|
|Operational|138366|
|  operating| 27854|
|Acquisition|  7024|
|     closed|  6434|
|    acquire|  2370|
|        ipo|   893|
|      close|   688|
|        ipO|   536|
|        IPO|   481|
|   acquired|   445|
|       ipos|   117|
|       NULL|     5|
+-----------+------+



In [19]:
# operate, operates, Operational, operating -> operating
# ipo, ipO, IPO, ipos -> ipo
# Acquisition, acquire, acquired -> acquired
# closed, close -> closed
# after counting the numbers, we see that operating is the highest so I will add the NULL values to operating.
# otherwise method added for just in case.

objects = objects.withColumn("status",when(lower(col("status")).isin("operate", "operates", "operational", "operating"), "operating")
                             .when(lower(col("status")).isin("ipo", "ip0", "ipos"), "ipo")
                             .when(lower(col("status")).isin("acquisition", "acquire", "acquired"), "acquired")
                             .when(lower(col("status")).isin("closed", "close"), "closed")
                             .when(col("status").isNull(), "operating").otherwise("operating"))

# Display count of each status
objects.groupBy("status").count().orderBy("count", ascending=False).show()

+---------+------+
|   status| count|
+---------+------+
|operating|443663|
| acquired|  9839|
|   closed|  7122|
|      ipo|  2027|
+---------+------+



## 2.2 Clean Funding_rounds table

1. Cleaning raised_amount_usd column:

       The raised_amount_usd has values in inconsistent formatting with special characters. Remove those and create a consistent formatting of values.

2. Filling NULL values in raised_amount_usd column:

        Calculate the average 'raised_amount_usd' for each 'funding_round_type'. For each funding type, replace missing or null values in the raised_amount_usd column with the corresponding average amount raised for that funding type.

3. Create new column 'total_amount_raised' by aggregating the 'raised_amount_usd' column based on 'object_id'.

3. Display the dataframe


Hint: Use groupBy() and mean()

In [20]:
#Fix values for column: raised_amount_usd

# lets check first "raised_amount_usd" column
funding_rounds.select("raised_amount_usd").show(50)
# we can see "$" and "," and they are causing inconsistency.


+-----------------+
|raised_amount_usd|
+-----------------+
|        8500000.0|
|         500000.0|
|       12700000.0|
|       27500000.0|
|       10500000.0|
|        1500000.0|
|       10000000.0|
|        1500000.0|
|        6300000.0|
|          12000.0|
|          40000.0|
|       40000000.0|
|       13500000.0|
|        3710000.0|
|        5700000.0|
|       13200000.0|
|        1500000.0|
|      45000000.00|
|        3800000.0|
|      $8500000.00|
|       12500000.0|
|       10520000.0|
|        6500000.0|
|        2500000.0|
|        5500000.0|
|        7000000.0|
|       $9,000,000|
|        5000000.0|
|        2250000.0|
|       12500000.0|
|       25000000.0|
|        1000000.0|
|       30000000.0|
|        5000000.0|
|       26000000.0|
|        6850000.0|
|       6000000.00|
|        6000000.0|
|       25000000.0|
|        700000.00|
|        4000000.0|
|       20,000,000|
|        7500000.0|
|       12500000.0|
|        7000000.0|
|        3000000.0|
|        1500000.0|


In [21]:
funding_rounds.printSchema()
# raised_amount_usd' s type string so we have to convert it to numeric type.

root
 |-- object_id: string (nullable = true)
 |-- funded_at: date (nullable = true)
 |-- funding_round_type: string (nullable = true)
 |-- funding_round_code: string (nullable = true)
 |-- raised_amount_usd: string (nullable = true)
 |-- is_first_round: integer (nullable = true)
 |-- is_last_round: integer (nullable = true)



In [22]:
# fix values for raised_amount_usd
funding_rounds = funding_rounds.withColumn("raised_amount_usd",regexp_replace(col("raised_amount_usd"), "[$,]", ""))

# cast to double
funding_rounds = funding_rounds.withColumn("raised_amount_usd",col("raised_amount_usd").cast(DoubleType()))

funding_rounds.select("raised_amount_usd").show(10)

+-----------------+
|raised_amount_usd|
+-----------------+
|        8500000.0|
|         500000.0|
|           1.27E7|
|           2.75E7|
|           1.05E7|
|        1500000.0|
|            1.0E7|
|        1500000.0|
|        6300000.0|
|          12000.0|
+-----------------+
only showing top 10 rows



In [23]:
# Aggregate raised_amount_usd

# calculate average raised_amount_usd per funding_round_type
avg_by_type = funding_rounds.groupBy("funding_round_type").agg(avg("raised_amount_usd").alias("avg_amount"))

# join back and fill nulls with the average for that type
funding_rounds = funding_rounds.join(avg_by_type, on="funding_round_type", how="left")

funding_rounds = funding_rounds.withColumn("raised_amount_usd",when(col("raised_amount_usd").isNull(), col("avg_amount")).otherwise(col("raised_amount_usd")))

# drop the avg_amount column since we don't need it anymore
funding_rounds = funding_rounds.drop("avg_amount")


In [24]:
#display dataframe
funding_rounds.select("funding_round_type", "raised_amount_usd").show(10)


+------------------+-----------------+
|funding_round_type|raised_amount_usd|
+------------------+-----------------+
|          series-b|        8500000.0|
|             angel|         500000.0|
|          series-a|           1.27E7|
|          series-b|           2.75E7|
|          series-b|           1.05E7|
|          series-a|        1500000.0|
|          series-b|            1.0E7|
|          series-a|        1500000.0|
|          series-a|        6300000.0|
|             angel|          12000.0|
+------------------+-----------------+
only showing top 10 rows



In [25]:
# aggregate raised_amount_usd by object_id
total_amount = funding_rounds.groupBy("object_id").agg(sum("raised_amount_usd").alias("total_amount_raised"))

# Join back to funding_rounds
funding_rounds = funding_rounds.join(total_amount, on="object_id", how="left")

In [26]:
funding_rounds.show(10)

+---------+------------------+----------+------------------+-----------------+--------------+-------------+-------------------+
|object_id|funding_round_type| funded_at|funding_round_code|raised_amount_usd|is_first_round|is_last_round|total_amount_raised|
+---------+------------------+----------+------------------+-----------------+--------------+-------------+-------------------+
|      c:4|          series-b|2006-12-01|                 b|        8500000.0|             0|            0|              4.5E7|
|      c:5|             angel|2004-09-01|             angel|         500000.0|             0|            1|           2.4257E9|
|      c:5|          series-a|2005-05-01|                 a|           1.27E7|             0|            0|           2.4257E9|
|      c:5|          series-b|2006-04-01|                 b|           2.75E7|             0|            0|           2.4257E9|
|   c:7299|          series-b|2006-05-01|                 b|           1.05E7|             0|           

## 3. Print duplicate rows count and remove those rows from each dataframe

In [27]:
#Drop Duplicates

# print duplicate counts
print("objects duplicates:", objects.count() - objects.dropDuplicates().count())
print("acquisitions duplicates:", acquisitions.count() - acquisitions.dropDuplicates().count())
print("funding_rounds duplicates:", funding_rounds.count() - funding_rounds.dropDuplicates().count())
print("investments duplicates:", investments.count() - investments.dropDuplicates().count())
print("relationships duplicates:", relationships.count() - relationships.dropDuplicates().count())

# remove duplicates
objects = objects.dropDuplicates()
acquisitions = acquisitions.dropDuplicates()
funding_rounds = funding_rounds.dropDuplicates()
investments = investments.dropDuplicates()
relationships = relationships.dropDuplicates()


objects duplicates: 0


acquisitions duplicates: 0


funding_rounds duplicates: 80
investments duplicates: 102


relationships duplicates: 0


## 4. Splitting the Objects Table

The object dataset consists of information about startups, including details about their products and the nature of their relationships with financial organizations and persons.

We would like to create different entities based on the entity_type. Using these entities we will derive new features which will be later used in training.



#### **Question: Divide the 'objects' dataset into four distinct datasets based on the 'entity_type' column. Also display count of rows for each dataset.**

Hint: Use filter() method to filter rows using the given condition.

In [28]:
# lets check first which entity_types exist
objects.groupBy("entity_type").count().orderBy("count", ascending=False).show()

+------------+------+
| entity_type| count|
+------------+------+
|      Person|226708|
|     Company|196553|
|     Product| 27738|
|FinancialOrg| 11652|
+------------+------+



In [29]:
#Create startups
startups = objects.filter(col("entity_type") == "Company")
print("startups count:", startups.count())


startups count: 196553


In [30]:
#Create financial_org
financial_org = objects.filter(col("entity_type") == "FinancialOrg")
print("financial_org count:", financial_org.count())


financial_org count: 11652


In [31]:
# Create products
products = objects.filter(col("entity_type") == "Product")
print("products count:", products.count())


products count: 27738


In [32]:
# Create persons
persons = objects.filter(col("entity_type") == "Person")
print("persons count:", persons.count())


persons count: 226708


## 5. Derive new features

This part will require you to apply joins and derive aggregated features by grouping data.


Hint: Use groupBy() to group and agg() to aggregate over dataframe, alias() to give new name to column, join() to join tables if required.

### 5.1

You are provided with dataset named 'acquisitions'.

    i) The column 'acquiring_object_id' indicates the entity that has made the acquisition. Group the data based on the 'acquiring_object_id'. For each group, count the total number of acquisitions made. Rename the aggregated count to 'num_acquisitions'. 
    Your final output should be a DataFrame named 'acquiring' that lists each 'acquiring_object_id' alongside the corresponding 'num_acquisitions'. Display the dataframe.

 
    ii) The column 'acquired_object_id' specifies the entity that was acquired. Group the dataset by 'acquired_object_id' to organize the data by each entity that has been acquired. Compute the count of acquisitions for each group. Rename the aggregated count to 'have_been_acquired'. 
    Your final output should be a DataFrame named 'acquired' that lists each 'acquired_object_id' alongside the corresponding 'have_been_acquired'. Display the dataframe.

In [33]:
# create num_acquisitions
acquisitions.show(5)


+----+--------------+-------------------+------------------+---------+------------+-------------------+-----------+
|  id|acquisition_id|acquiring_object_id|acquired_object_id|term_code|price_amount|price_currency_code|acquired_at|
+----+--------------+-------------------+------------------+---------+------------+-------------------+-----------+
|  88|           116|              c:350|            c:1611|     cash|       5.0E7|                USD| 1999-06-01|
|2345|          2643|            c:36704|           c:36703|     cash|      5.25E8|                USD| 2009-12-01|
|3745|          4125|             c:8249|           c:61846|     cash|     1.375E7|                USD| 2010-11-17|
|4892|          5316|            c:56939|           c:71564|     cash|   1800000.0|                USD| 2005-08-08|
|6680|          7307|            c:67920|          c:162797|     cash|         0.0|                USD| 2012-06-27|
+----+--------------+-------------------+------------------+---------+--

In [34]:
# display acquiring df

# num_acquisitions
acquiring = acquisitions.groupBy("acquiring_object_id").agg(count("acquisition_id").alias("num_acquisitions"))

acquiring.show(10)

+-------------------+----------------+
|acquiring_object_id|num_acquisitions|
+-------------------+----------------+
|              c:396|               7|
|            c:63963|               1|
|           c:184544|               1|
|            c:71686|               1|
|            c:68400|               2|
|            c:31480|               1|
|            c:64864|               2|
|            c:14334|               3|
|            c:18125|               1|
|            c:29403|               1|
+-------------------+----------------+
only showing top 10 rows



In [35]:
# have_been_acquired

acquired = acquisitions.groupBy("acquired_object_id").agg(count("acquisition_id").alias("have_been_acquired"))


In [36]:
# display acquired df

acquired.show(10)


+------------------+------------------+
|acquired_object_id|have_been_acquired|
+------------------+------------------+
|          c:102575|                 1|
|           c:21061|                 1|
|           c:58286|                 1|
|           c:31079|                 1|
|           c:43688|                 1|
|          c:257665|                 1|
|           c:38468|                 1|
|          c:163730|                 1|
|           c:26490|                 2|
|           c:20310|                 1|
+------------------+------------------+
only showing top 10 rows



### 5.2

    Utilize the 'investments' and 'financial_org' datasets to identify how many times each entity has been financed by financial organizations.
  
    Your final output should be a DataFrame named 'finorgs' that lists each 'funded_object_id' alongside the corresponding 'fin_org_financed'. Display the dataframe.

    

In [37]:
# create finorgs

finorgs = investments.join(financial_org,investments["investor_object_id"] == financial_org["id"],"inner").groupBy("funded_object_id").agg(count("investor_object_id").alias("fin_org_financed"))


In [38]:
# display finorgs

finorgs.show(10)


+----------------+----------------+
|funded_object_id|fin_org_financed|
+----------------+----------------+
|         c:28035|               1|
|         c:48759|               6|
|        c:218090|               4|
|         c:26086|               9|
|         c:26248|               3|
|         c:12836|               1|
|         c:79778|               1|
|         c:79120|               6|
|         c:53141|              11|
|         c:31480|               6|
+----------------+----------------+
only showing top 10 rows



### 5.3

    Determine the number of investments made by individuals in various entities using the 'investments' and 'persons' datasets. 

    Your final output should be a DataFrame named 'num_persons' that lists each 'funded_object_id' alongside the corresponding 'person_financed'. Display the dataframe.

In [39]:
# create num_persons

num_persons = investments.join(persons,investments["investor_object_id"] == persons["id"],"inner").groupBy("funded_object_id").agg(count("investor_object_id").alias("person_financed"))


In [40]:
# display num_persons

num_persons.show(10)


+----------------+---------------+
|funded_object_id|person_financed|
+----------------+---------------+
|         c:41922|              3|
|         c:68400|              4|
|         c:25418|              2|
|         c:57411|              6|
|         c:62247|              2|
|         c:53141|              5|
|         c:10751|              2|
|        c:187163|              3|
|         c:13086|              1|
|          c:1639|              4|
+----------------+---------------+
only showing top 10 rows



### 5.4

    Calculate how many times each entity has been financed by startups using the 'investments' and 'startups' datasets.

    Your final output should be a DataFrame named 'nstartup' that lists each 'funded_object_id' alongside the corresponding 'startup_financed'. Display the dataframe.

In [41]:
# create nstartup

nstartup = investments.join(startups,investments["investor_object_id"] == startups["id"],"inner").groupBy("funded_object_id").agg(count("investor_object_id").alias("startup_financed"))

In [42]:
# display nstartup

nstartup.show(10)

+----------------+----------------+
|funded_object_id|startup_financed|
+----------------+----------------+
|        c:173274|               1|
|        c:237076|               1|
|        c:196703|               1|
|         c:15709|               2|
|        c:160902|               2|
|        c:240778|               1|
|         c:75318|               1|
|        c:211860|               1|
|         c:17455|               1|
|         c:75245|               1|
+----------------+----------------+
only showing top 10 rows



### 5.5

    Determine the number of products associated with each parent entity using the 'products' dataset.

      Your final output should be a DataFrame named 'nproducts' that lists each 'parent_id' alongside the corresponding 'num_products'. Display the dataframe.

In [43]:
# create nproducts

nproducts = products.groupBy("parent_id").agg(count("id").alias("num_products"))


In [44]:
# display nproducts

nproducts.show(10)


+---------+------------+
|parent_id|num_products|
+---------+------------+
|  c:21156|           9|
|  c:12836|           1|
|  c:19003|          12|
|  c:11928|           4|
|  c:12550|          10|
|  c:11251|          12|
|   c:6758|          11|
|  c:14821|           3|
| c:241019|           2|
|  c:80940|           6|
+---------+------------+
only showing top 10 rows



## 6. Joins

You will join all the tables created above and create a final dataset. The name of the final dataset will be train_data. 

Joins in spark: https://spark.apache.org/docs/3.1.2/api/python/reference/api/pyspark.sql.DataFrame.join.html

 ### 6.1
 
    Combine the 'startups' and 'funding_rounds' datasets to analyze funding details for each startup. Your output should be a DataFrame named 'train_data'. Print columns of final dataset.

In [45]:
# lets check the id's first if they are matching.

print("startups id:")
startups.select("id").show(3)

print("funding_rounds object_id:")
funding_rounds.select("object_id").show(3)

startups id:


+--------+
|      id|
+--------+
|c:248727|
|c:248862|
| c:24935|
+--------+
only showing top 3 rows

funding_rounds object_id:
+---------+
|object_id|
+---------+
|  c:21084|
|  c:57411|
|   c:5032|
+---------+
only showing top 3 rows



In [46]:
# join startups and funding_rounds

train_data = startups.join(funding_rounds,startups["id"] == funding_rounds["object_id"],"left").drop("object_id")
print(train_data.columns)

['id', 'entity_type', 'entity_id', 'parent_id', 'name', 'category_code', 'status', 'country_code', 'state_code', 'city', 'region', 'investment_rounds', 'invested_companies', 'first_funding_at', 'last_funding_at', 'funding_rounds', 'funding_total_usd', 'participants', 'milestones', 'relationships', 'age_of_company', 'funding_round_type', 'funded_at', 'funding_round_code', 'raised_amount_usd', 'is_first_round', 'is_last_round', 'total_amount_raised']


### 6.2

    Join the 'acquiring' and 'acquired' DataFrames with the existing 'train_data' DataFrame to integrate data on acquisitions. Print columns of final dataset. You don't require the id through which you are joining these tables. Drop those columns after joining.

In [47]:
# lets check for acquiring and acquired's object ids.

print("acquiring acquiring_object_id:")
acquiring.select("acquiring_object_id").show(3)

print("acquired acquired_object_id:")
acquired.select("acquired_object_id").show(3)

acquiring acquiring_object_id:
+-------------------+
|acquiring_object_id|
+-------------------+
|              c:396|
|            c:10346|
|            c:18125|
+-------------------+
only showing top 3 rows

acquired acquired_object_id:
+------------------+
|acquired_object_id|
+------------------+
|            c:7601|
|           c:18076|
|           c:20444|
+------------------+
only showing top 3 rows



In [48]:
# acquiring
train_data = train_data.join(acquiring,train_data["id"] == acquiring["acquiring_object_id"],"left").drop("acquiring_object_id")

# acquired
train_data = train_data.join(acquired,train_data["id"] == acquired["acquired_object_id"],"left").drop("acquired_object_id")

print(train_data.columns)


['id', 'entity_type', 'entity_id', 'parent_id', 'name', 'category_code', 'status', 'country_code', 'state_code', 'city', 'region', 'investment_rounds', 'invested_companies', 'first_funding_at', 'last_funding_at', 'funding_rounds', 'funding_total_usd', 'participants', 'milestones', 'relationships', 'age_of_company', 'funding_round_type', 'funded_at', 'funding_round_code', 'raised_amount_usd', 'is_first_round', 'is_last_round', 'total_amount_raised', 'num_acquisitions', 'have_been_acquired']


### 6.3

    Join the 'train_data' DataFrame by merging it with 'finorgs', 'num_persons', 'nstartup', and 'nproducts' DataFrames based on relevant ID matches and streamline the merged dataset by removing redundant columns during each join operation. Print columns of final dataset.

In [49]:
# lets check for 'finorgs', 'num_persons', 'nstartup', and 'nproducts' relevant ids.

print("finorgs funded_object_id:")
finorgs.select("funded_object_id").show(3)

print("num_persons funded_object_id:")
num_persons.select("funded_object_id").show(3)

print("nstartup funded_object_id:")
nstartup.select("funded_object_id").show(3)

print("nproducts parent_id:")
nproducts.select("parent_id").show(3)

finorgs funded_object_id:


+----------------+
|funded_object_id|
+----------------+
|         c:28035|
|         c:48759|
|        c:218090|
+----------------+
only showing top 3 rows

num_persons funded_object_id:


+----------------+
|funded_object_id|
+----------------+
|        c:286183|
|         c:53141|
|         c:57411|
+----------------+
only showing top 3 rows

nstartup funded_object_id:


+----------------+
|funded_object_id|
+----------------+
|        c:171476|
|         c:26490|
|        c:160902|
+----------------+
only showing top 3 rows

nproducts parent_id:


+---------+
|parent_id|
+---------+
|  c:14171|
|  c:11516|
|  c:14821|
+---------+
only showing top 3 rows



In [50]:
# lets join finorgs, num_persons, nstartup, nproducts with train_data

train_data = train_data.join(finorgs,train_data["id"] == finorgs["funded_object_id"],"left").drop("funded_object_id")

train_data = train_data.join(num_persons,train_data["id"] == num_persons["funded_object_id"],"left").drop("funded_object_id")

train_data = train_data.join(nstartup,train_data["id"] == nstartup["funded_object_id"],"left").drop("funded_object_id")

train_data = train_data.join(nproducts,train_data["id"] == nproducts["parent_id"],"left").drop("parent_id")

# lets print train_data columns
print(train_data.columns)

['id', 'entity_type', 'entity_id', 'name', 'category_code', 'status', 'country_code', 'state_code', 'city', 'region', 'investment_rounds', 'invested_companies', 'first_funding_at', 'last_funding_at', 'funding_rounds', 'funding_total_usd', 'participants', 'milestones', 'relationships', 'age_of_company', 'funding_round_type', 'funded_at', 'funding_round_code', 'raised_amount_usd', 'is_first_round', 'is_last_round', 'total_amount_raised', 'num_acquisitions', 'have_been_acquired', 'fin_org_financed', 'person_financed', 'startup_financed', 'num_products']


### 6.4

We have compiled a comprehensive dataset that includes information on financial organization investments, individual investments, startup investments, and product counts under each entity.

    Display the count of train_data. Also, display the schema of train dataset i.e the column names with corresponding data types. 

In [51]:
# Write your code here

# train_data counts
print("train_data count:", train_data.count())

# train_data schema
train_data.printSchema()


train_data count: 217013
root
 |-- id: string (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- entity_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- status: string (nullable = false)
 |-- country_code: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- investment_rounds: integer (nullable = true)
 |-- invested_companies: integer (nullable = true)
 |-- first_funding_at: date (nullable = true)
 |-- last_funding_at: date (nullable = true)
 |-- funding_rounds: double (nullable = true)
 |-- funding_total_usd: double (nullable = true)
 |-- participants: string (nullable = true)
 |-- milestones: double (nullable = true)
 |-- relationships: double (nullable = true)
 |-- age_of_company: double (nullable = true)
 |-- funding_round_type: string (nullable = true)
 |-- funded_at: date (nullable = true)
 |-- funding_round

In [52]:
train_data.show(10)

26/02/21 17:24:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+-----------+---------+------------------+-------------+---------+------------+----------+-------------+---------------+-----------------+------------------+----------------+---------------+--------------+-----------------+------------+----------+-------------+------------------+------------------+----------+------------------+-----------------+--------------+-------------+-------------------+----------------+------------------+----------------+---------------+----------------+------------+
|      id|entity_type|entity_id|              name|category_code|   status|country_code|state_code|         city|         region|investment_rounds|invested_companies|first_funding_at|last_funding_at|funding_rounds|funding_total_usd|participants|milestones|relationships|    age_of_company|funding_round_type| funded_at|funding_round_code|raised_amount_usd|is_first_round|is_last_round|total_amount_raised|num_acquisitions|have_been_acquired|fin_org_financed|person_financed|startup_financed|num_

In [53]:
# Run this cell to remove unecessary columns:
columns_to_drop = ['parent_id', 'entity_id', 'closed_at', 'funded_object_id']
train_data = train_data.drop(*columns_to_drop)

### 6.5 Converting datatypes of feature columns

Convert all the numeric data columns to float and print them

In [54]:
from pyspark.sql.functions import col

# Columns to convert to float
columns_to_convert = [
    'investment_rounds',
    'invested_companies',
    'funding_rounds',
    'relationships',
    'total_amount_raised',
    'num_acquisitions',
    'fin_org_financed',
    'num_products',
    'startup_financed',
    'person_financed',
    'funding_total_usd',
    'participants',
    'age_of_company',
    'raised_amount_usd',
    'have_been_acquired'
]

In [55]:
# Convert specified columns to float

for col_name in columns_to_convert:
    train_data = train_data.withColumn(col_name, col(col_name).cast(FloatType()))

# print the converted columns
train_data.select(columns_to_convert).printSchema()


root
 |-- investment_rounds: float (nullable = true)
 |-- invested_companies: float (nullable = true)
 |-- funding_rounds: float (nullable = true)
 |-- relationships: float (nullable = true)
 |-- total_amount_raised: float (nullable = true)
 |-- num_acquisitions: float (nullable = true)
 |-- fin_org_financed: float (nullable = true)
 |-- num_products: float (nullable = true)
 |-- startup_financed: float (nullable = true)
 |-- person_financed: float (nullable = true)
 |-- funding_total_usd: float (nullable = true)
 |-- participants: float (nullable = true)
 |-- age_of_company: float (nullable = true)
 |-- raised_amount_usd: float (nullable = true)
 |-- have_been_acquired: float (nullable = true)



### 6.6 Transform the 'id' column in the 'train_data' DataFrame by removing any 'c:' prefix.

In [56]:
from pyspark.sql.functions import col, regexp_replace

columns_to_convert = ['id']

In [57]:
train_data = train_data.withColumn("id",regexp_replace(col("id"), "c:", ""))

# lets check if we replace c: with ""
train_data.select("id").show(5)

+------+
|    id|
+------+
|248727|
|248862|
| 24935|
|249461|
|249586|
+------+
only showing top 5 rows



## Q7. Spark Mlib: Classification

MLlib(Main Guide): https://spark.apache.org/docs/latest/ml-guide.html

MLlib (DataFrame-based): https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html




In this part, you will perform multiclass classification task using Apache Spark's scalable machine learning library(Spark Mlib).

Predict the status of startups by performing a classification analysis on the categorical dependent variable 'Status'

The dependent variable is a categorical one, made up of 4 non-orderable levels, indicating the STATUS of each startup. These levels are:

    CLOSED : failed startup

    ACQUIRED : acquired startup

    IPO : listed startup

    OPERATING : startup not acquired or listed

### 7.1 Fill Missing Values for categorical columns

    Perform Mode imputation for categorical columns like 'category_code', 'country_code'. 

    Mode imputation: Replace missing values with the mode (the most frequently occurring value) of the column.


Hint: Order your column in descending order according to frequency and get the first value.
Methods Used: groupBy(), orderBy(), count(), desc(), first()

In [58]:
# lets check category_code
print("category_code frequencies:")
train_data.groupBy("category_code").count().orderBy("count", ascending=False).show(5)

# lets check country_code
print("country_code frequencies:")
train_data.groupBy("country_code").count().orderBy("count", ascending=False).show(5)

category_code frequencies:


+-------------+-----+
|category_code|count|
+-------------+-----+
|         NULL|73571|
|     software|20704|
|          web|16500|
|        other|13831|
|    ecommerce| 9915|
+-------------+-----+
only showing top 5 rows

country_code frequencies:


+------------+------+
|country_code| count|
+------------+------+
|        NULL|108844|
|         USA| 67901|
|         GBR|  8313|
|         CAN|  4245|
|         IND|  4065|
+------------+------+
only showing top 5 rows



In [59]:
# Write your code here

# get mode for category_code (first non-null most frequent value)
category_mode = train_data.groupBy("category_code").count().orderBy(desc("count")).filter(col("category_code").isNotNull()).first()[0]
print("category_code mode:", category_mode)

# get mode for country_code
country_mode = train_data.groupBy("country_code").count().orderBy(desc("count")).filter(col("country_code").isNotNull()).first()[0]
print("country_code mode:", country_mode)

# fill nulls with mode
train_data = train_data.fillna({"category_code": category_mode, "country_code": country_mode})


category_code mode: software


country_code mode: USA


### 7.2 Convert categorical columns to numeric using String Indexer

    String Indexer assigns a unique integer based on label frequencies, with the most frequent label getting index 0.

StringIndexer: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html

In [60]:
# lets check first which columns are categorical
train_data.printSchema()

root
 |-- id: string (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category_code: string (nullable = false)
 |-- status: string (nullable = false)
 |-- country_code: string (nullable = false)
 |-- state_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- investment_rounds: float (nullable = true)
 |-- invested_companies: float (nullable = true)
 |-- first_funding_at: date (nullable = true)
 |-- last_funding_at: date (nullable = true)
 |-- funding_rounds: float (nullable = true)
 |-- funding_total_usd: float (nullable = true)
 |-- participants: float (nullable = true)
 |-- milestones: double (nullable = true)
 |-- relationships: float (nullable = true)
 |-- age_of_company: float (nullable = true)
 |-- funding_round_type: string (nullable = true)
 |-- funded_at: date (nullable = true)
 |-- funding_round_code: string (nullable = true)
 |-- raised_amount_usd: float (nullable = 

In [61]:
# Write your code here

categorical_columns = ["category_code", "country_code", "funding_round_type", "funding_round_code", "status"]

indexers = [StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep").fit(train_data) for c in categorical_columns]
pipeline = Pipeline(stages=indexers)
train_data = pipeline.fit(train_data).transform(train_data)




In [62]:
# If you've renamed the original columns, then drop them
train_data = train_data.drop(*categorical_columns)

# lets check the columns again
print(train_data.columns)

['id', 'entity_type', 'name', 'state_code', 'city', 'region', 'investment_rounds', 'invested_companies', 'first_funding_at', 'last_funding_at', 'funding_rounds', 'funding_total_usd', 'participants', 'milestones', 'relationships', 'age_of_company', 'funded_at', 'raised_amount_usd', 'is_first_round', 'is_last_round', 'total_amount_raised', 'num_acquisitions', 'have_been_acquired', 'fin_org_financed', 'person_financed', 'startup_financed', 'num_products', 'category_code_index', 'country_code_index', 'funding_round_type_index', 'funding_round_code_index', 'status_index']


 ### 7.3 Fill the missing values for numerical columns

    First identify the numerical columns. Use the imputer method to fill the missing values using 'mean' strategy for these columns.

Imputer(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.Imputer.html


In [63]:
train_data.printSchema()

root
 |-- id: string (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- investment_rounds: float (nullable = true)
 |-- invested_companies: float (nullable = true)
 |-- first_funding_at: date (nullable = true)
 |-- last_funding_at: date (nullable = true)
 |-- funding_rounds: float (nullable = true)
 |-- funding_total_usd: float (nullable = true)
 |-- participants: float (nullable = true)
 |-- milestones: double (nullable = true)
 |-- relationships: float (nullable = true)
 |-- age_of_company: float (nullable = true)
 |-- funded_at: date (nullable = true)
 |-- raised_amount_usd: float (nullable = true)
 |-- is_first_round: integer (nullable = true)
 |-- is_last_round: integer (nullable = true)
 |-- total_amount_raised: float (nullable = true)
 |-- num_acquisitions: float (nullable = true)
 |-- have_been_acquired: float (nu

In [64]:
# Write your code here 

numerical_columns = [
    'investment_rounds', 'invested_companies', 'funding_rounds',
    'funding_total_usd', 'participants', 'milestones', 'relationships',
    'age_of_company', 'raised_amount_usd', 'is_first_round', 'is_last_round',
    'total_amount_raised', 'num_acquisitions', 'have_been_acquired',
    'fin_org_financed', 'person_financed', 'startup_financed', 'num_products',
    'category_code_index', 'country_code_index', 'funding_round_type_index',
    'funding_round_code_index'
]

imputer = Imputer(strategy="mean", inputCols=numerical_columns, outputCols=numerical_columns)
train_data = imputer.fit(train_data).transform(train_data)


In [65]:
# drop unnecessary columns
columns_to_drop = ["id", "participants", "is_first_round", "is_last_round", "acquiring_object_id", "acquired_object_id", "funding_round_type", "funded_at", "funding_round_code", "raised_amount_usd","object_id"]
train_data = train_data.drop(*columns_to_drop)

In [66]:
train_data.columns

['entity_type',
 'name',
 'state_code',
 'city',
 'region',
 'investment_rounds',
 'invested_companies',
 'first_funding_at',
 'last_funding_at',
 'funding_rounds',
 'funding_total_usd',
 'milestones',
 'relationships',
 'age_of_company',
 'total_amount_raised',
 'num_acquisitions',
 'have_been_acquired',
 'fin_org_financed',
 'person_financed',
 'startup_financed',
 'num_products',
 'category_code_index',
 'country_code_index',
 'funding_round_type_index',
 'funding_round_code_index',
 'status_index']

### 7.4 Feature vector

    
    Use Pyspark ML's VectorAssembler() which is a feature transformer to convert the feature columns to single column vector. Index the target variable 'status'.

VectorAssembler(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html

In [67]:
train_data.printSchema()

root
 |-- entity_type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- investment_rounds: float (nullable = true)
 |-- invested_companies: float (nullable = true)
 |-- first_funding_at: date (nullable = true)
 |-- last_funding_at: date (nullable = true)
 |-- funding_rounds: float (nullable = true)
 |-- funding_total_usd: float (nullable = true)
 |-- milestones: double (nullable = true)
 |-- relationships: float (nullable = true)
 |-- age_of_company: float (nullable = true)
 |-- total_amount_raised: float (nullable = true)
 |-- num_acquisitions: float (nullable = true)
 |-- have_been_acquired: float (nullable = true)
 |-- fin_org_financed: float (nullable = true)
 |-- person_financed: float (nullable = true)
 |-- startup_financed: float (nullable = true)
 |-- num_products: float (nullable = true)
 |-- category_code_index: double (nullable = false)
 |-- co

In [68]:
# after looking at the schema, lets drop features aren't numeric; entity_type, name, state_code, city, region, first_funding_at, last_funding_at
train_data = train_data.drop("entity_type", "name", "state_code", "city", "region", "first_funding_at", "last_funding_at")

# define feature columns (everything except target variable)
feature_columns = [c for c in train_data.columns if c != "status_index"]
print("Feature columns:", feature_columns)

# assemble features
assembler = VectorAssembler(inputCols=feature_columns, outputCol="feature_vector")
train_data = assembler.transform(train_data)

train_data.select("feature_vector", "status_index").show(5)




Feature columns: ['investment_rounds', 'invested_companies', 'funding_rounds', 'funding_total_usd', 'milestones', 'relationships', 'age_of_company', 'total_amount_raised', 'num_acquisitions', 'have_been_acquired', 'fin_org_financed', 'person_financed', 'startup_financed', 'num_products', 'category_code_index', 'country_code_index', 'funding_round_type_index', 'funding_round_code_index']


+--------------------+------------+
|      feature_vector|status_index|
+--------------------+------------+
|[0.0,0.0,3.0,3.97...|         0.0|
|[0.0,0.0,3.0,3.97...|         0.0|
|[0.0,0.0,3.0,3.97...|         0.0|
|[0.0,0.0,0.0,0.0,...|         1.0|
|[0.0,0.0,0.0,0.0,...|         0.0|
+--------------------+------------+
only showing top 5 rows



### 7.5 Model selection with 5-fold cross-validation (Random Forest vs Logistic Regression)

Split the dataset into **train/test**. On the **training split only**, run **5-fold cross-validation** (Spark ML `CrossValidator`) for **two models** and pick the best model/hyperparameters based on the CV metric. Then evaluate the selected best model on the held-out **test** split (no CV-on-test) and report test accuracy.

RandomForest(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.RandomForestClassifier.

LogisticRegression(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html

CrossValidator(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html

ParamGridBuilder(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.ParamGridBuilder.html

Pipeline(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html

MulticlassClassificationEvaluator(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.MulticlassClassificationEvaluator.html


In [69]:
# split into train and test
train, test = train_data.randomSplit([0.7, 0.3], seed=42)
print("Train size:", train.count())
print("Test size:", test.count())


Train size: 152114


Test size: 64899


In [70]:
# Random Forest
rf = RandomForestClassifier(labelCol="status_index", featuresCol="feature_vector")

rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

evaluator = MulticlassClassificationEvaluator(labelCol="status_index", predictionCol="prediction", metricName="accuracy")

rf_cv = CrossValidator(estimator=rf, estimatorParamMaps=rf_paramGrid, evaluator=evaluator, numFolds=5)
rf_cv_model = rf_cv.fit(train)

print("Best RF CV accuracy:", max(rf_cv_model.avgMetrics))

26/02/21 17:26:47 WARN DAGScheduler: Broadcasting large task binary with size 1013.0 KiB
26/02/21 17:26:57 WARN DAGScheduler: Broadcasting large task binary with size 1177.3 KiB
26/02/21 17:26:58 WARN DAGScheduler: Broadcasting large task binary with size 1582.6 KiB
26/02/21 17:26:59 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/02/21 17:27:01 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/02/21 17:27:03 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/02/21 17:27:25 WARN DAGScheduler: Broadcasting large task binary with size 1022.7 KiB
26/02/21 17:27:37 WARN DAGScheduler: Broadcasting large task binary with size 1171.3 KiB
26/02/21 17:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1586.2 KiB
26/02/21 17:27:39 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/02/21 17:27:40 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/02/21 17:27:42 WARN DAGScheduler:

Best RF CV accuracy: 0.984209260060727


In [71]:
# Logistic Regression
lr = LogisticRegression(labelCol="status_index", featuresCol="feature_vector", maxIter=10)

lr_paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .build()

lr_cv = CrossValidator(estimator=lr, estimatorParamMaps=lr_paramGrid, evaluator=evaluator, numFolds=5)
lr_cv_model = lr_cv.fit(train)

print("Best LR CV accuracy:", max(lr_cv_model.avgMetrics))

26/02/21 17:30:27 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Best LR CV accuracy: 0.9256984942152732


### 7.6 Evaluation

    Evaluate the performance of a machine learning model on a multi-class classification problem. Print out precision and recall for each class identified in the model's predictions.

MulticlassMetrics(): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.mllib.evaluation.MulticlassMetrics.html

In [72]:
# pick best model and evaluate on test
rf_test_acc = evaluator.evaluate(rf_cv_model.transform(test))
lr_test_acc = evaluator.evaluate(lr_cv_model.transform(test))

print("RF Test Accuracy:", rf_test_acc)
print("LR Test Accuracy:", lr_test_acc)

if rf_test_acc > lr_test_acc:
    print("Best model: Random Forest")
else:
    print("Best model: Logistic Regression")


26/02/21 17:32:22 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB


RF Test Accuracy: 0.9841754110232823
LR Test Accuracy: 0.9248832801738085
Best model: Random Forest
